# FIP 01 — Movement (motion energy) × RPE / value coding

Extends `fip_00_explore.ipynb`'s single-session FIP + motion-energy (ME) pipeline with the
analysis used for the FIP-only tonic-value / phasic-RPE figure (baseline AUC by consecutive
reward streak, outcome traces by RPE bin), applied to motion energy across all example FIP
channels. Two figures:

1. Session-averaged movement over time per RPE bin
2. Z-scored AUC per consecutive R-/R+ (`num_reward_past`)

Recipe traced directly from `rachel-analysis-utils`, `aind-dynamic-foraging-basic-analysis`,
`aind-dynamic-foraging-data-utils`, and (for the plotting conventions) the `DA_phasic_tonic`
repo — see the implementation plan (`fip_01` plan, 2026-09-14) for the full trace and the
deviations reviewed against that source. Setup cells below are copied from
`fip_00_explore.ipynb` rather than factored into a shared module — see `fip_todo.md`.

A third figure (NE-only onsets aligned to movement) originally lived here; it moved to
`fip_02_ne_only_events.ipynb`, which compares NE and DA transients directly rather than
treating all NE onsets alike.

## Imports & setup

In [ ]:
import os
import glob
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import fastparquet  # noqa: F401  # load_nwb_list uses pd.read_parquet(engine="fastparquet")

# Rachel's lab utilities (installed / mounted in the Code Ocean capsule).
from rachel_analysis_utils import nwb_utils as r_utils

# AIND event-alignment primitive (peri-event windowing) and the FIP enrichment helpers
# (z-scoring, per-trial windowing, tonic/baseline removal) -- the same functions Rachel's own
# pipeline uses, not a reimplementation of them.
from aind_dynamic_foraging_data_utils import alignment, enrich_dfs

# Upstream PSTH-by-category plotter, reused for the RPE-binned movement panel (Figure 1).
from aind_dynamic_foraging_basic_analysis.plot import plot_fip as pf

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## Data loading

In [ ]:
# Same asset as fip_00_explore.ipynb.
plot_loc = "/root/capsule/data/DA_NE_4channels/"

assert os.path.isdir(plot_loc), (
    f"{plot_loc} not found. Attach asset 6babbf3d-6970-4456-aab5-d730ed57c269 "
    "(saved parquet hierarchy) to this capsule, or fix the path."
)
print("plot_loc =", plot_loc)

In [ ]:
# load_nwb_list returns df_bg (background-coefficient csvs) as a 4th value; unused here
# (same as df_sess/df_slope) but must be unpacked.
nwb_list_raw, df_sess, df_slope, df_bg = r_utils.load_nwb_list(plot_loc, load_fip=True)
print(f"Loaded {len(nwb_list_raw)} session(s)")

### Curation

Same as `fip_00_explore.ipynb` section 3, including the patch for
`data_curation_helpers.apply_curation_nwb_list`'s missing private-helper imports (an upstream
bug -- see that notebook for the full explanation).

In [ ]:
USE_CURATION = True
CURATION_FILE = "DA_NE_4channel_datacuration_firstpass"

# apply_curation_nwb_list lives in data_curation_helpers, not nwb_utils.
from rachel_analysis_utils import data_curation_helpers as r_curation

# Upstream bug: data_curation_helpers.apply_curation_nwb_list calls private helpers that it
# never imports -- they live in nwb_utils and work fine there, so patch them in. Gated so this
# becomes a no-op once the helpers are imported upstream.
for _helper in ("_parse_session_id", "_actual_map_for_session", "_get_df_trials_col_mapping",
                "_map_event_to_intended_measurement", "_apply_channel_drops_to_nwb"):
    if not hasattr(r_curation, _helper):
        setattr(r_curation, _helper, getattr(r_utils, _helper))

json_path = None
try:
    import rachel_analysis_utils
    pkg_dir = os.path.dirname(rachel_analysis_utils.__file__)
    cand = os.path.join(pkg_dir, "data_curation", CURATION_FILE + ".json")
    if os.path.exists(cand):
        json_path = cand
except Exception:
    pass
if json_path is None:
    hits = glob.glob("/src/**/data_curation/" + CURATION_FILE + ".json", recursive=True)
    assert hits, "Could not locate " + CURATION_FILE + ".json"
    json_path = hits[0]

with open(json_path, "r") as fh:
    df_curation = json.load(fh)
print("Using curation:", json_path)

# drop_borderline only controls whether a SECOND list (curated_with_borderline) gets built --
# nwb_list itself is identical either way. That second list is never used here, but
# apply_curation_nwb_list still deep-copies + processes every session to build it when True,
# doubling memory for nothing. False skips that pass entirely.
nwb_list, _nwb_list_curated_unused = r_curation.apply_curation_nwb_list(
    nwb_list_raw, df_curation, drop_borderline=False
)
print(f"Curated -> {len(nwb_list)} session(s) kept.")

# nwb_list_raw is a full second copy of the pre-curation dataset and isn't referenced again
# below -- free it now instead of letting it sit in RAM for the rest of the kernel. Matters
# most with fip_00_explore.ipynb and this notebook open in separate kernels at the same time,
# each independently holding a full copy.
del nwb_list_raw, _nwb_list_curated_unused
import gc
gc.collect()

## Data processing

### Select a session and build the signal table

Same example session as `fip_00_explore.ipynb` (`SESSION_IDX = 0`).

In [ ]:
SESSION_IDX = 0
nwb = nwb_list[SESSION_IDX]
df_fip = nwb.df_fip
assert (df_fip.groupby("event")["timestamps"].diff().dropna() >= 0).all(), \
    "df_fip timestamps not sorted within an event — sort before use"
df_trials = getattr(nwb, "df_trials", None)

ALIGN_COL = "goCue_start_time_in_session"
assert df_trials is not None and ALIGN_COL in df_trials.columns, (
    f"Need '{ALIGN_COL}' on df_trials (first-go-cue-zeroed clock). Available: "
    f"{None if df_trials is None else list(df_trials.columns)}")

# enrich_fip_in_df_trials's windowing (below) assumes goCue_start_time_in_trial == 0 for every
# trial (go cue defines trial-local time zero) -- verify rather than assume.
assert "goCue_start_time_in_trial" in df_trials.columns, "Need 'goCue_start_time_in_trial'"
assert np.allclose(df_trials["goCue_start_time_in_trial"].dropna(), 0.0), (
    "goCue_start_time_in_trial is not all zero -- enrich_fip_in_df_trials's per-trial window "
    "math (below) assumes the go cue is trial-local time zero. Re-check before proceeding.")

print("session_id:", nwb.session_id, "| df_fip", df_fip.shape, "| trials", df_trials.shape)

In [ ]:
def parse_event(name):
    """'G_1_dff-poly' -> ('G','1','dff-poly'); 'R_0' -> ('R','0','raw')."""
    parts = name.split("_")
    channel = parts[0]
    fiber = parts[1] if len(parts) > 1 else "?"
    variant = "_".join(parts[2:]) if len(parts) > 2 else "raw"
    return channel, fiber, variant


def get_trace(df_fip, event_name, data_col="data"):
    """(t, y) arrays for one df_fip series (asserted time-sorted per event)."""
    sub = df_fip[df_fip["event"] == event_name]
    return sub["timestamps"].to_numpy(), sub[data_col].to_numpy()


events = [e for e in sorted(df_fip["event"].unique()) if "pearson" not in e.lower()]
meta = pd.DataFrame([(e,) + parse_event(e) for e in events],
                    columns=["event", "channel", "fiber", "variant"])
if "intended_measurement" in df_fip.columns:
    ev2region = (df_fip.dropna(subset=["intended_measurement"])
                 .groupby("event")["intended_measurement"]
                 .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None))
    meta["region"] = meta["event"].map(ev2region)
print(f"{len(events)} FIP series (pearsonR excluded)")
meta

### Trial enrichment — `num_reward_past` and `RPE-binned3`

`rachel_analysis_utils.analysis_utils.enrich_df_trials` is the real source of both columns
(see plan). Its module previously failed to import on Python 3.9 (a since-apparently-fixed
nested-quote f-string); try it first, and fall back to a local reimplementation of just the two
columns we need if it still fails, so this notebook doesn't hard-depend on that being fixed.

In [ ]:
def _enrich_streaks_and_rpe_bins_fallback(df_trials):
    """3.9-safe reimplementation of the two rachel_analysis_utils.analysis_utils.enrich_df_trials
    columns this notebook needs (num_reward_past, RPE-binned3), matching its exact logic."""
    df = df_trials.copy()
    df["reward_all"] = df["earned_reward"].astype(float) + df["extra_reward"].astype(float)
    df["rewarded_prev"] = df["reward_all"].shift(1)
    df["num_reward_past"] = df.groupby(
        (df["rewarded_prev"] != df["reward_all"]).cumsum()).cumcount() + 1
    df.loc[df["reward_all"] == 0, "num_reward_past"] *= -1

    rpe_labels = [str(np.round(i, 2)) for i in np.arange(-1, 0.99, 1 / 3)]
    bins = np.arange(-1, 1.01, 1 / 3)
    bins[-1] = 1.001
    df["RPE-binned3"] = pd.cut(df["RPE_earned"], bins=bins, right=True, labels=rpe_labels)
    return df


try:
    from rachel_analysis_utils import analysis_utils as r_analysis
    df_trials = r_analysis.enrich_df_trials(df_trials)
    print("Used rachel_analysis_utils.analysis_utils.enrich_df_trials directly.")
except Exception as e:
    print(f"enrich_df_trials unavailable ({type(e).__name__}: {e}); using local fallback "
          "for num_reward_past + RPE-binned3.")
    df_trials = _enrich_streaks_and_rpe_bins_fallback(df_trials)

nwb.df_trials = df_trials
RPE_binned3_label_names = df_trials["RPE-binned3"].cat.categories.astype(str).tolist()
print("RPE-binned3 categories:", RPE_binned3_label_names)
df_trials[["num_reward_past", "RPE-binned3"]].describe(include="all")

### Example signals

All three example FIP channels used in `fip_00_explore.ipynb` (NAc DA/dLight, PL/GCaMP,
NAc ACh/rAch), same `pick_example` selection logic.

In [ ]:
def pick_example(meta, df_fip, region_substr, prefer_variant="dff"):
    """Choose one df_fip 'event' whose region label matches `region_substr`.

    Prefers a dff variant, then the series with the most finite samples.
    """
    if "region" not in meta.columns or meta["region"].dropna().empty:
        raise RuntimeError(
            "No region labels on `meta` -- set USE_CURATION=True (section 3) with the "
            "DA_NE_4channel curation so df_fip carries intended_measurement.")
    cand = meta[meta["region"].fillna("").str.contains(region_substr, case=False, regex=False)]
    if cand.empty:
        raise ValueError("No FIP series with region matching %r. Have: %s"
                         % (region_substr, sorted(meta["region"].dropna().unique())))
    pref = cand[cand["variant"].str.contains(prefer_variant, case=False, regex=False)]
    pool = pref if not pref.empty else cand
    best = max(pool["event"], key=lambda ev: int(np.isfinite(get_trace(df_fip, ev)[1]).sum()))
    row = pool[pool["event"] == best].iloc[0]
    return {"event": best, "region": row["region"], "channel": row["channel"],
            "variant": row["variant"]}


# (region substring, plot colour) for each example signal type -- same as fip_00_explore.ipynb.
EXAMPLE_SPECS = [
    ("NAc DA (dLight)", "dLight", "#2ca02c"),
    ("PL (GCaMP)",      "Gcamp",  "#1f77b4"),
    ("NAc ACh (rAch)",  "rAch",   "#d62728"),
]

examples = []
for label, substr, color in EXAMPLE_SPECS:
    info = pick_example(meta, df_fip, substr)
    info["label"], info["color"] = label, color
    examples.append(info)

pd.DataFrame(examples)[["label", "event", "channel", "region", "variant", "color"]]

### Motion energy on the FIP clock

Same as `fip_00_explore.ipynb` sections 8a/8b.

In [ ]:
# --- Locate the behavior-video + motion-energy assets for THIS session ---
subject, date = nwb.session_id.split("_")[:2]

beh_dirs = [d for d in sorted(glob.glob(f"/root/capsule/data/behavior_{subject}_{date}_*"))
            if "motionenergy" not in d]
assert beh_dirs, f"No behavior-video asset for {subject}_{date} under /root/capsule/data"
video_csv = os.path.join(beh_dirs[0], "behavior-videos", "bottom_camera.csv")
assert os.path.exists(video_csv), f"missing {video_csv}"

me_dirs = sorted(glob.glob(f"/root/capsule/data/behavior_{subject}_{date}_*motionenergy*"))
assert me_dirs, f"No motion-energy asset for {subject}_{date}"
hits = glob.glob(os.path.join(me_dirs[0], "**", "bottom_camera_motion_energy_clean.npy"),
                 recursive=True)
assert hits, f"bottom_camera_motion_energy_clean.npy not found under {me_dirs[0]}"
me_path = hits[0]

print("video_csv:", video_csv)
print("me_path:  ", me_path)

In [ ]:
import sys, subprocess, importlib

try:
    from aind_dynamic_foraging_behavior_video_analysis import video_alignment as va
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/AllenNeuralDynamics/"
        "aind-dynamic-foraging-behavior-video-analysis.git@main"])
    importlib.invalidate_caches()
    from aind_dynamic_foraging_behavior_video_analysis import video_alignment as va

CAM_COLUMNS = list(va.DEFAULT_COLUMNS)


def motion_energy_to_session(me_path, video_csv, df_trials, go_cue_col="goCue_start_time_raw"):
    """Per-frame motion energy on the session clock (t=0 at first go cue).

    See fip_00_explore.ipynb for the full explanation of the padding/clock logic.
    Returns (t_session, me, offset).
    """
    me = np.load(me_path)
    cam = va.read_video_csv(video_csv, columns=CAM_COLUMNS)
    time_col = next(c for c in va.TIME_COLUMN_ALIASES if c in cam.columns)

    meta_path = me_path.replace("_motion_energy_clean.npy", "_me_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            me_meta = json.load(fh)
        pad = me_meta["n_me_frames"] == me_meta["n_frames_decoded"] - 1
    else:
        pad = (len(cam) - len(me)) == 1
    if pad:
        me = np.insert(me, 0, 0.0)

    if len(me) != len(cam):
        n = min(len(me), len(cam))
        print(f"WARNING: {len(me)} ME vs {len(cam)} CSV rows after pad; truncating to {n}")
        me, cam = me[:n], cam.iloc[:n]

    first_go_cue = float(df_trials.sort_values("trial")[go_cue_col].iloc[0])
    offset = va.compute_video_session_offset(video_csv, first_go_cue)
    first_frame = va.get_first_frame_behavior_time(video_csv)
    video_t = va.behavior_time_to_video_time(cam[time_col].to_numpy(float), first_frame)
    t_session = va.video_time_to_session_time(video_t, offset)
    return t_session, np.asarray(me, float), offset


t_me, me, offset = motion_energy_to_session(me_path, video_csv, nwb.df_trials)
print(f"offset = {offset:.3f} s | ME {t_me.min():.1f}..{t_me.max():.1f}s | "
      f"FIP {df_fip['timestamps'].min():.1f}..{df_fip['timestamps'].max():.1f}s")

### Helpers — z-scoring, onset detection, peri-event alignment

Same as `fip_00_explore.ipynb` (`norm_xcorr` omitted -- not used in this notebook).

In [ ]:
def zscore(y):
    """Z-score a 1D array, ignoring NaNs."""
    y = np.asarray(y, float)
    sd = np.nanstd(y)
    return (y - np.nanmean(y)) / sd if sd > 0 else y - np.nanmean(y)


def threshold_onsets(t, y, z_thresh=2.5, refractory=0.5, min_run=1):
    """Causal upward threshold crossings of the z-scored trace (no smoothing)."""
    z = zscore(np.asarray(y, float))
    above = z > z_thresh
    idx = np.where((~above[:-1]) & above[1:])[0] + 1
    if min_run > 1 and len(idx):
        idx = np.array([i for i in idx
                        if i + min_run <= len(above) and above[i:i + min_run].all()], dtype=int)
    times = np.asarray(t, float)[idx]
    if len(times):
        times = times[np.insert(np.diff(times) > refractory, 0, True)]
    return times


def peri_event(t, y, event_times, t_before=1.0, t_after=3.0, fs=20, censor=True,
               censor_times=None):
    """Align signal (t, y) to event_times via the AIND ETR primitive."""
    s = pd.DataFrame({"timestamps": np.asarray(t, float),
                      "data": np.asarray(y, float)}).dropna()
    return alignment.event_triggered_response(
        data=s, t="timestamps", y="data",
        event_times=np.asarray(event_times, float),
        t_before=t_before, t_after=t_after,
        output_sampling_rate=fs, output_format="tidy", censor=censor,
        censor_times=censor_times)

### Attach ME + run the real baseline/tonic-normalization pipeline

`attach_me_to_df_fip` stores **raw** motion energy (not pre-z-scored, unlike
`fip_00_explore.ipynb`'s version) so ME flows through `zscore_fip` /
`enrich_fip_in_df_trials` / `remove_tonic_df_fip` exactly like a real FIP channel -- these are
Rachel's actual functions (`aind_dynamic_foraging_data_utils.enrich_dfs`), channel-agnostic, run
once over the whole `df_fip` (every real channel + `"ME"` together). Produces `data_z` (per
`(session, event)` z-score) and, per trial, `data_z_{event}_baseline` (mean over the 1s
immediately before that trial's go cue) and `data_z_{event}_norm` (baseline-subtracted).

In [ ]:
def attach_me_to_df_fip(df_fip, t_me, me, ses_idx, event_name="ME"):
    """Append RAW motion energy to df_fip as a pseudo-channel (event='ME'), so it goes
    through zscore_fip/enrich_fip_in_df_trials/remove_tonic_df_fip identically to a real FIP
    channel (those functions z-score `data` themselves -- passing pre-z-scored data here would
    silently double-process it)."""
    me_rows = pd.DataFrame({"timestamps": np.asarray(t_me, float),
                            "data": np.asarray(me, float),
                            "event": event_name,
                            "intended_measurement": "motion_energy",
                            "ses_idx": ses_idx})
    return pd.concat([df_fip, me_rows], ignore_index=True)


df_fip = attach_me_to_df_fip(df_fip, t_me, me, nwb.session_id)

# enrich_fip_in_df_trials z-scores internally (via zscore_fip) before windowing, so a separate
# upfront zscore_fip call isn't needed.
df_fip_z, df_trials_fip = enrich_dfs.enrich_fip_in_df_trials(df_fip, df_trials)
df_fip_tonic, df_trials, df_trials_fip = enrich_dfs.remove_tonic_df_fip(
    df_fip_z, df_trials, df_trials_fip)

nwb.df_fip = df_fip_z
nwb.df_trials = df_trials

# One pass produces baseline/norm columns for every channel in df_fip at once (channel-agnostic
# -- see title-cell plan note); check them all here, right after the pipeline runs, rather than
# failing deep inside a plotting loop later.
_baseline_cols = [f"data_z_{ex['event']}_baseline" for ex in examples] + ["data_z_ME_baseline"]
for _col in _baseline_cols:
    assert _col in df_trials.columns, f"missing {_col}"
print("baseline columns ready:", _baseline_cols)

## Figure 1 — session-averaged movement per RPE bin, all signals

One panel per signal (the 3 example FIP channels + motion energy). Same function, alignment
event, time window, censoring, and mako-by-ascending-bin color convention as `DA_phasic_tonic`'s
`PAC_2026.ipynb` recipe for the DA-channel version of this panel -- now looped over every
signal instead of just ME.

In [ ]:
rpe_dict = {
    label: df_trials.loc[df_trials["RPE-binned3"] == label, "choice_time_in_session"].dropna().to_numpy()
    for label in RPE_binned3_label_names
}
for label, times in rpe_dict.items():
    print(f"  RPE {label}: {len(times)} trials")

rpe_colors = dict(zip(RPE_binned3_label_names,
                      sns.color_palette("mako", len(RPE_binned3_label_names)).as_hex()))

plot_signals = examples + [{"label": "Motion energy", "event": "ME", "color": "#555555"}]

fig, axes = plt.subplots(1, len(plot_signals), figsize=(4 * len(plot_signals), 3.6), sharex=True)
for ax, sig in zip(np.atleast_1d(axes), plot_signals):
    pf.plot_fip_psth_compare_alignments(
        nwb, rpe_dict, channel=sig["event"], tw=[-1, 2], censor=True,
        data_column="data_z", extra_colors=rpe_colors, ax=ax, fig=fig)
    ax.set_title(sig["label"], loc="left", fontsize=10, color=sig["color"])
    ax.set_ylabel("signal (z)")
fig.suptitle(f"Signals aligned to choice, by RPE bin — {nwb.session_id}")
fig.tight_layout()
plt.show()

## Figure 2 — z-scored AUC per consecutive R-/R+, all signals

`num_reward_past_baseline = num_reward_past.shift(1)` pairs each trial's baseline value with
the reward-streak count *before* it was recorded (the `power_analysis.ipynb` /
`BWNM_explore_data.ipynb` convention -- mathematically the same pairing
`foraging_summary_plots.py::plot_baseline_corr` gets by shifting baseline the other way). Same
`sns.barplot(..., palette="vlag", hue=..., dodge=False)` recipe as that function's "Column 0",
applied to the real `remove_tonic_df_fip` baseline columns -- one panel per signal (the 3
example FIP channels + motion energy). `remove_tonic_df_fip` already ran over every channel in
`df_fip` in one pass (Attach ME section above), so every `data_z_{event}_baseline` column is
already there; no extra enrichment needed here.

In [ ]:
df_trials["num_reward_past_baseline"] = df_trials["num_reward_past"].shift(1)
df_bl = df_trials.query("num_reward_past_baseline > -7 and num_reward_past_baseline < 7")

baseline_signals = (
    [(ex["label"], f"data_z_{ex['event']}_baseline", ex["color"]) for ex in examples]
    + [("Motion energy", "data_z_ME_baseline", "#555555")]
)

fig, axes = plt.subplots(1, len(baseline_signals), figsize=(4 * len(baseline_signals), 4))
for ax, (label, col, color) in zip(np.atleast_1d(axes), baseline_signals):
    assert col in df_trials.columns, f"missing {col}"
    sns.barplot(x="num_reward_past_baseline", y=col, data=df_bl,
               palette="vlag", hue="num_reward_past_baseline", dodge=False, ax=ax)
    ax.legend().remove()
    ax.set_xlabel("# consecutive R-/R+ trials\n(num_reward_past, shifted)")
    ax.set_ylabel("baseline (z)")
    ax.set_title(label, color=color)
fig.suptitle(f"Baseline AUC vs. consecutive reward streak — {nwb.session_id}")
fig.tight_layout()
plt.show()